# What makes a restaurant succeed in Bangalore?

An analysis of ~12,000 restaurants listed on Zomato in Bangalore.

**Questions**
1. Does offering online ordering or table booking go with better ratings?
2. Which areas and cuisines dominate, and which are rated highest?
3. Do more expensive restaurants get better ratings?
4. Can we predict whether a restaurant will be rated 4.0+ from its basic attributes?
5. Where are the gaps: area + cuisine combinations with engaged customers but few competitors?

**Data:** [Zomato Bangalore Restaurants (Kaggle)](https://www.kaggle.com/datasets/himanshupoddar/zomato-bangalore-restaurants). Download `zomato.csv` into `data/`.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src import analysis, model
from src.clean import clean, load_raw

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.2f}".format)

## 1. Load and inspect the raw data

In [ ]:
raw = load_raw("../data/zomato.csv")
print(raw.shape)
raw.head()

In [ ]:
# What do the messy columns look like?
print(raw["rate"].value_counts().head(10))
print(raw["approx_cost(for two people)"].unique()[:15])
print("Missing values:\n", raw.isna().sum().sort_values(ascending=False).head(8))

Things to fix:
- `rate` is text like `"4.1/5"`, with `"NEW"` and `"-"` for unrated restaurants.
- Cost is text with commas (`"1,200"`).
- The same restaurant appears several times, once per listing category (Delivery, Dine-out, ...).

In [ ]:
print("Rows:", len(raw), "| unique name+address:", raw[["name", "address"]].drop_duplicates().shape[0])

## 2. Clean

Cleaning logic lives in `src/clean.py` so the notebook, report script and dashboard all use the same steps.

In [ ]:
df = clean(raw)
print(df.shape)
df[["name", "location", "rating", "cost_for_two", "online_order", "book_table", "primary_cuisine"]].head()

In [ ]:
analysis.overview(df)

## 3. Rating distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(df["rating"].dropna(), bins=30, ax=ax)
ax.set(title="Distribution of ratings", xlabel="Rating", ylabel="Restaurants")
plt.show()

## 4. Does online ordering or table booking go with better ratings?

Ratings aren't normally distributed, so I use a Mann-Whitney U test instead of a t-test.

In [ ]:
for flag in ["online_order", "book_table"]:
    r = analysis.compare_groups(df, flag)
    print(f"{flag}: with={r['with']:.2f} (n={r['n_with']}), without={r['without']:.2f} "
          f"(n={r['n_without']}), diff={r['difference']:+.2f}, p={r['p_value']:.2g}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, col in zip(axes, ["online_order", "book_table"]):
    sns.boxplot(data=df, x=col, y="rating", ax=ax)
    ax.set_title(col.replace("_", " ").title())
plt.show()

**Interpretation:** _write 2-3 sentences here once you've seen the numbers._ Remember this is correlation, not causation: restaurants that take bookings tend to be more expensive sit-down places, which may explain part of the difference.

## 5. Areas and cuisines

In [ ]:
analysis.by_area(df)

In [ ]:
analysis.top_cuisines(df)

In [ ]:
analysis.by_restaurant_type(df)

## 6. Does price go with rating?

In [ ]:
price = analysis.rating_by_price(df)
price

In [ ]:
ax = price["share_rated_4_plus"].plot.bar(rot=0, figsize=(7, 4))
ax.set(title="Share rated 4.0+ by cost for two (INR)", ylabel="Share")
plt.show()

In [ ]:
df[["cost_for_two", "rating", "votes", "n_cuisines"]].corr(method="spearman")

## 7. Can we predict a 4.0+ rating?

I leave **votes** out of the features. Votes accumulate alongside the rating, so a brand-new restaurant wouldn't have them, and including them would leak information about the target.

In [ ]:
result = model.train(df)
print(f"ROC-AUC: {result['roc_auc']:.3f}   (share of 4.0+ in test set: {result['baseline_positive_rate']:.0%})")
print(result["report"])

In [ ]:
ax = result["importance"].sort_values().plot.barh(figsize=(8, 5))
ax.set(title="Feature importance", xlabel="Importance")
plt.show()

## 8. Opportunity gaps

Area + cuisine combinations where restaurants get lots of votes on average (customers are engaged) but there are few competitors. `opportunity_score = avg votes per restaurant / number of restaurants`.

In [ ]:
analysis.opportunity_gaps(df)

## 9. Summary and recommendations

_Fill this in after running the notebook. Aim for 3-5 bullet points a restaurant owner or Zomato's partnerships team could act on, for example:_
- _Where to open a new restaurant, and which cuisine_
- _Whether to invest in online ordering or table booking_
- _What price band performs best_

**Limitations:** the data is a 2019 snapshot; ratings are self-selected; area names in the data overlap (e.g. listing city vs location).